# Klasik Görüntü İşleme ile Hücre Segmentasyonu ve Sayımı – Grup Çalışması
**Veri seti:** Cell Tracking Challenge – Fluo-N2DH-GOWT1 · **Ders:** Görüntü İşleme (Yüksek Lisans Ara Sınav Projesi)

**Hazırlayanlar:** Süleyman Balsak (254312022, seed 14048) · Seda Erten (254312008, seed 14034) ·
İlkin Urcan (244329078, seed 31104) · Aslıhan Semiz Varlı (254329024, seed 31050)

### Nasıl çalıştırılır: iki aşama
| Aşama | Ne yapılır | Sonuç |
|---|---|---|
| **1** | *Çalışma zamanı → Tümünü çalıştır* | Bölüm 0–8 çalışır, **kilit** dosyaları yazılır, Bölüm 9'daki kontrol notebook'u **durdurur** |
| ara | `final_pipeline.json` + `test_predictions_locked.csv` (ve deney/doğrulama dosyaları) GitHub'a **ayrı commit** ile gönderilir | Test tahminleri test referansı görülmeden kayıt altına girer |
| **2** | Yeni bir Colab oturumunda yine *Tümünü çalıştır* | Kilit doğrulanır (tahminler birebir aynı olmalı), test ve zamansal analiz çalışır |

Test referansları (sekans 02) kodla korunur: kilit dosyası **commit edilmeden** okunamaz.

Üyelerin deney listeleri `src/experiments/<üye>.py` dosyalarındadır; her üye yalnızca kendi dosyasını düzenler.

## 0. Kurulum

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/James007Bond-code/goruntu-isleme-hucre-sayimi-grup-odevi.git"
REPO_NAME = "goruntu-isleme-hucre-sayimi-grup-odevi"
if not os.path.isdir("src"):
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
sys.path.insert(0, os.getcwd())
print("Çalışma klasörü:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-5"], capture_output=True, text=True).stdout)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import download_dataset, load_image, read_gt_count, describe_sequence, lock_is_committed
from src.selection import build_selection, load_selection, dev_pool_excluding_all_val, leaky_val_frames
from src.pipeline import BASELINE, E11_SULEYMAN, make, param_str
from src.evaluation import (evaluate, evaluate_method, warmup, run_experiments, rank_candidates,
                            lock_predictions, method_complexity, count_metrics, save_json)
from src.experiments import load as load_experiments

for d in [C.FIG_DIR, C.FIG_IMG_DIR, C.RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
SEEDS = list(C.SEEDS.values())
for k, m in C.MEMBERS.items():
    print(f"{m['name']:20s} {m['student_id']}  seed = {m['student_id'][-5:]} + 2026 = {m['seed']}")

## 1. Veri setinin indirilmesi ve ön incelemesi (Ödev Bölüm 2)
Bu adımda referans etiketleri okunmaz.

In [ ]:
n01, n02 = download_dataset()
print(f"Sekans 01: {n01} kare | Sekans 02: {n02} kare")
df_over = pd.DataFrame(describe_sequence("01") + describe_sequence("02"))
df_over.to_csv("data_overview.csv", index=False)
print(df_over.groupby("sequence")[["height", "width", "min", "max", "mean", "std", "p99"]]
      .agg(["min", "max"]).round(1).to_string())
print("Veri tipleri:", df_over["dtype"].unique())

fig, ax = plt.subplots(figsize=(7, 3))
for seq, col in [("01", "#1E88E5"), ("02", "#E53935")]:
    im = load_image(seq, f"t{n01 // 2:03d}.tif")
    ax.hist(im.ravel(), bins=128, range=(0, 255), histtype="step", color=col, label=f"sekans {seq}", log=True)
ax.set_xlabel("Yoğunluk"); ax.set_ylabel("Piksel sayısı (log)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Örnek karelerin yoğunluk histogramı", fontsize=10)
plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/fig0_histograms.png", dpi=200); plt.show()

## 2. Kare seçimi (Ödev Bölüm 3)
Seçim ilk commit'te kaydedildi (`seeds/<seed>/selected_frames.csv`). Burada yeniden üretilir ve birebir aynı olduğu doğrulanır.

In [ ]:
sels = {}
for s in SEEDS:
    saved = load_selection(s)
    regen, _ = build_selection(s, n01, n02)
    assert saved.equals(regen.astype(saved.dtypes.to_dict())), f"seed {s}: kayıtlı seçim uyuşmuyor!"
    sels[s] = saved
df_sel = pd.concat(sels.values(), ignore_index=True)
df_sel.to_csv("selected_frames.csv", index=False)          # Tablo 1 (4 seed birleşik)
print("TABLO 1 – Seçilen kareler")
print(df_sel.groupby(["seed", "group"])["frame_idx"].apply(list).to_string())
print("\nSeed'ler arası çakışma:")
print(pd.read_csv(os.path.join(C.RESULTS_DIR, "seed_overlap.csv")).query("n_ortak > 0").to_string(index=False))

## 3. Referans hücre sayıları (Ödev Bölüm 4)
Yalnızca **sekans 01** (geliştirme + doğrulama). `TRA` etiketlerindeki sıfır dışı farklı kimlikler sayılır.

In [ ]:
refs = {}
for s, df in sels.items():
    dv = df[df.group.isin(["Development", "Validation"])].copy()
    dv["gt_count"] = [read_gt_count(r.sequence, r.frame_idx) for r in dv.itertuples()]
    assert dv["gt_count"].notna().all(), f"seed {s}: eksik referans"
    dv.to_csv(os.path.join(C.seed_dir(s), "dev_val_references.csv"), index=False)
    refs[s] = dv
pd.concat(refs.values()).to_csv("dev_val_references.csv", index=False)
print(pd.concat(refs.values()).groupby(["seed", "group"])["gt_count"].agg(["min", "max", "mean"]).round(1).to_string())
warmup(load_image("01", sels[C.BASE_SEED].iloc[0].filename))

## 4. Üye deneyleri (Ödev Bölüm 5–10)
Her üye kendi seed'inin **geliştirme** kümesinde kontrollü deneylerini çalıştırır (her deney referanstan tek faktörle ayrılır).
Baseline dışındaki en düşük geliştirme MAE'li 3 deney aday olur; kendi **doğrulama** kümesinde en iyisi üyenin yöntemi olur
(eşitlikte RMSE, karmaşıklık ve geliştirme sıralaması; süre raporlanır ama ölçüm gürültülü olduğu için eşitlik bozmada kullanılmaz). Histogram matching referansı yalnızca üyenin kendi geliştirme kümesinden seçilir.

In [ ]:
def run_member(member):
    seed = C.SEEDS[member]
    family, experiments = load_experiments(member)
    dv = refs[seed]
    dev_df = dv[dv.group == "Development"].reset_index(drop=True)
    val_df = dv[dv.group == "Validation"].reset_index(drop=True)
    hm_ref_name = dev_df.loc[0, "filename"]
    hm_ref = load_image("01", hm_ref_name)
    print(f"=== {C.MEMBERS[member]['name']} | seed {seed} | {family}")

    df_exp, df_pf = run_experiments(experiments, dev_df, hm_ref, member=member, seed=seed)
    df_exp.to_csv(os.path.join(C.seed_dir(seed), f"experiments_{member}.csv"), index=False)
    df_pf.to_csv(os.path.join(C.seed_dir(seed), f"experiments_per_frame_{member}.csv"), index=False)
    print(df_exp[["Deney_ID", "Aciklama", "Degisen", "Dev_MAE", "Dev_RMSE", "Dev_Bias", "Sure_ms"]].to_string(index=False))

    cfg_by_id = {eid: cfg for eid, *_, cfg in experiments}
    top3 = df_exp[df_exp["Deney_ID"] != experiments[0][0]].sort_values(["Dev_MAE", "Dev_RMSE", "Deney_ID"]).head(3)
    cands = {f"{r.Deney_ID}: {r.Aciklama}": cfg_by_id[r.Deney_ID] for r in top3.itertuples()}
    df_val = rank_candidates(cands, val_df, hm_ref)
    df_val.insert(0, "Uye", member); df_val.insert(1, "Seed", seed)
    df_val.to_csv(os.path.join(C.seed_dir(seed), f"validation_{member}.csv"), index=False)
    print("\nDoğrulama:"); print(df_val[["Pipeline", "Validation_MAE", "Validation_RMSE", "Ortalama_Sure_ms", "Secim"]].to_string(index=False))

    best = df_val.loc[0, "Pipeline"]; cfg = cands[best]
    save_json({"name": f"M_{member}", "owner": member, "seed": seed, "experiment": best, "config": cfg,
               "hm_ref": hm_ref_name if cfg.get("intensity") == "hist_match" else None,
               "validation_own_seed": {k: float(df_val.loc[0, k]) for k in ["Validation_MAE", "Validation_RMSE"]}},
              os.path.join(C.seed_dir(seed), f"method_{member}.json"))
    print(f"\n→ Üyenin yöntemi: {best} | {param_str(cfg)}")

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.bar(df_exp["Deney_ID"], df_exp["Dev_MAE"].clip(upper=20), color="#1E88E5")
    for i, v in enumerate(df_exp["Dev_MAE"]):
        ax.text(i, min(v, 20) + 0.3, f"{v:.1f}", ha="center", fontsize=8)
    ax.set_ylabel("Geliştirme MAE (20'de kırpıldı)"); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"{C.MEMBERS[member]['name']} – seed {seed} – kontrollü deneyler", fontsize=10)
    plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/experiments_{member}_{seed}.png", dpi=200); plt.show()
    return df_exp, df_val

### 4.1 Süleyman Balsak – seed 14048 – CLAHE + Gauss + adaptif eşik

In [ ]:
MEMBER = "suleyman"
df_exp_suleyman, df_val_suleyman = run_member(MEMBER)

### 4.2 Seda Erten – seed 14034 – yoğunluk dönüşümleri ve filtreler

In [ ]:
MEMBER = "seda"
df_exp_seda, df_val_seda = run_member(MEMBER)

### 4.3 İlkin Urcan – seed 31104 – morfoloji ve watershed

In [ ]:
MEMBER = "ilkin"
df_exp_ilkin, df_val_ilkin = run_member(MEMBER)

### 4.4 Aslıhan Semiz Varlı – seed 31050 – K-means ve LoG

In [ ]:
MEMBER = "aslihan"
df_exp_aslihan, df_val_aslihan = run_member(MEMBER)

In [ ]:
df_exp_all = pd.concat([df_exp_suleyman, df_exp_seda, df_exp_ilkin, df_exp_aslihan], ignore_index=True)
df_exp_all.to_csv("experiments.csv", index=False)                       # Tablo 2 (tüm üyeler)
pd.concat([df_val_suleyman, df_val_seda, df_val_ilkin, df_val_aslihan]).to_csv(
    os.path.join(C.RESULTS_DIR, "member_validation_results.csv"), index=False)
print(f"Toplam kontrollü deney: {len(df_exp_all)} (4 üye)")
methods = {k: json.load(open(os.path.join(C.seed_dir(s), f"method_{k}.json"))) for k, s in C.SEEDS.items()}

## 5. Çapraz seed doğrulama matrisi
Satır: yöntem, sütun: doğrulama seed'i. Yöntem sahibinin geliştirme karesi olan doğrulama kareleri o hücrede değerlendirilmez (`n_kare`).

In [ ]:
def val_frames(seed, exclude=()):
    v = refs[seed][refs[seed].group == "Validation"]
    return v[~v.frame_idx.isin(exclude)].reset_index(drop=True)

def owner_seeds(method):
    if "ensemble" in method:
        return [m["seed"] for m in method["ensemble"]]
    return [method["seed"]] if "seed" in method else []

def cross_seed_rows(method):
    rows = []
    for s in SEEDS:
        excl = sorted(set().union(*[leaky_val_frames(sels, o, s) for o in owner_seeds(method)])) if owner_seeds(method) else []
        m = evaluate_method(val_frames(s, excl), method)
        rows.append({"val_seed": s, "MAE": m["MAE"], "RMSE": m["RMSE"], "Bias": m["Bias"], "Sure_ms": m["Sure_ms"],
                     "n_kare": 5 - len(excl), "haric": " ".join(map(str, excl))})
    return rows

BASELINE_M = {"name": "baseline", "config": BASELINE, "hm_ref": None}
df_cross = pd.DataFrame([{"yontem": k, **r} for k, m in [("baseline", BASELINE_M)] + list(methods.items())
                         for r in cross_seed_rows(m)])
df_cross.to_csv(os.path.join(C.RESULTS_DIR, "cross_seed_validation.csv"), index=False)
mat = df_cross.pivot(index="yontem", columns="val_seed", values="MAE")[SEEDS]
mat["ortalama"] = mat[SEEDS].mean(axis=1); mat["std"] = mat[SEEDS].std(axis=1, ddof=0)
print("Doğrulama MAE – çapraz seed matrisi"); print(mat.round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.6))
im = ax.imshow(mat[SEEDS].to_numpy(), cmap="viridis_r")
ax.set_xticks(range(len(SEEDS)), SEEDS); ax.set_yticks(range(len(mat)), mat.index)
for i in range(len(mat)):
    for j in range(len(SEEDS)):
        ax.text(j, i, f"{mat.iloc[i, j]:.1f}", ha="center", va="center", color="w", fontsize=8)
ax.set_xlabel("Doğrulama seed'i"); ax.set_title("Çapraz seed doğrulama MAE", fontsize=10)
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/cross_seed_matrix.png", dpi=200); plt.show()

## 6. 5. yöntem: dayanıklı pipeline (Ödev Bölüm 11)
- **Ortak geliştirme havuzu:** 4 seed'in geliştirme kareleri; herhangi bir seed'in doğrulama karesiyle birebir aynı olanlar hariç.
- Birleşik aday (CLAHE + Gauss + adaptif eşik + watershed) bu havuzda küçük bir taramayla ayarlanır.
- Adaylar: 4 üye yöntemi + birleşik aday + 4 üye yönteminin medyan topluluğu.
- **Seçim kuralı (önceden sabit):** 4 seed doğrulama MAE ortalaması; eşitlikte seed'ler arası std, RMSE, karmaşıklık, aday sırası (ölçülen süre tekrarlanabilir olmadığı için kullanılmaz).

In [ ]:
pool, removed = dev_pool_excluding_all_val(sels)
pool_df = pd.DataFrame({"sequence": "01", "frame_idx": pool, "filename": [f"t{i:03d}.tif" for i in pool]})
pool_df["gt_count"] = [read_gt_count("01", i) for i in pool]
print(f"Ortak geliştirme havuzu: {len(pool)} kare | çıkarılan: {removed}")

COMBO_BASE = make(E11_SULEYMAN, separation="watershed", ws_min_dist=16)
GRID = [make(COMBO_BASE, adapt_c=c, ws_min_dist=d) for c in (-8, -12, -16) for d in (12, 16, 20)]
tune = []
for cfg in GRID:
    m = evaluate(pool_df, cfg)
    tune.append({"Parametreler": param_str(cfg), "Pool_MAE": round(m["MAE"], 3), "Pool_RMSE": round(m["RMSE"], 3), "cfg": cfg})
df_tune = pd.DataFrame(tune).sort_values(["Pool_MAE", "Pool_RMSE"], kind="stable").reset_index(drop=True)
df_tune.drop(columns="cfg").to_csv(os.path.join(C.RESULTS_DIR, "method5_tuning_pool.csv"), index=False)
print(df_tune.drop(columns="cfg").head(5).to_string(index=False))

M5_CANDIDATES = {
    **{f"M_{k}": m for k, m in methods.items()},
    "M5_combo": {"name": "M5_combo", "config": df_tune.loc[0, "cfg"], "hm_ref": None},
    "M5_ensemble_median": {"name": "M5_ensemble_median", "ensemble": list(methods.values()), "agg": "median"},
}
rows = []
for order, (name, meth) in enumerate(M5_CANDIDATES.items()):
    p = pd.DataFrame(cross_seed_rows(meth))
    rows.append({"Pipeline": name, "Ort_Val_MAE": round(p.MAE.mean(), 3), "Std_Val_MAE": round(p.MAE.std(ddof=0), 3),
                 "Ort_Val_RMSE": round(p.RMSE.mean(), 3), "Ort_Sure_ms": round(p.Sure_ms.mean(), 1),
                 "Karmasiklik": method_complexity(meth), "Aday_Sirasi": order + 1, **{f"MAE_{s}": round(v, 2) for s, v in zip(p.val_seed, p.MAE)}})
df_m5 = pd.DataFrame(rows).sort_values(["Ort_Val_MAE", "Std_Val_MAE", "Ort_Val_RMSE", "Karmasiklik", "Aday_Sirasi"]).reset_index(drop=True)
df_m5["Secim"] = ["SEÇİLDİ (5. yöntem)"] + ["elendi"] * (len(df_m5) - 1)
df_m5.to_csv("validation_results.csv", index=False)                    # Tablo 3
print("TABLO 3 – Aday pipeline'ların 4 seed doğrulama performansı"); print(df_m5.to_string(index=False))
M5_NAME = df_m5.loc[0, "Pipeline"]

## 7. 🔒 Kilit: nihai yöntem ve test tahminleri (Ödev Bölüm 12)
Bu hücre test referanslarını **okumaz**. Kilit dosyası zaten varsa (2. aşama) yeniden üretilen tahminlerin birebir aynı olduğu doğrulanır.

In [ ]:
FINAL = {"baseline": BASELINE_M, **{f"M_{k}": m for k, m in methods.items()}, "M5": M5_CANDIDATES[M5_NAME]}
final_json = {"selection_rule": "4 seed doğrulama MAE ortalaması; eşitlikte std, RMSE, karmaşıklık, aday sırası",
              "method5_name": M5_NAME, "methods": FINAL, "seeds": C.SEEDS, "seen_test_seeds": sorted(C.SEEN_TEST_SEEDS)}
if os.path.exists(C.FINAL_PIPELINE_FILE):
    old = json.load(open(C.FINAL_PIPELINE_FILE, encoding="utf-8"))
    assert json.dumps(old, sort_keys=True, default=float) == json.dumps(json.loads(json.dumps(final_json, default=float)), sort_keys=True), \
        "UYARI: final_pipeline.json kilitli sürümden farklı!"
    print("final_pipeline.json mevcut ve birebir aynı.")
else:
    save_json(final_json, C.FINAL_PIPELINE_FILE)
test_df = pd.concat([sels[s][sels[s].group == "Test"] for s in SEEDS]).reset_index(drop=True)
locked = lock_predictions(test_df, FINAL, C.LOCK_FILE)
print(f"5. yöntem: {M5_NAME}")

## 8. ⛔ Kilit kontrolü
`final_pipeline.json` ve `test_predictions_locked.csv` GitHub'a commit edilmeden aşağıdaki bölümler çalışmaz.

**1. aşamadaysanız:** Notebook burada durur. Bir sonraki hücredeki zip'i indirip dosyaları GitHub'a commit edin
(ya da zip'i grup sorumlusuna gönderin). Sonra yeni bir Colab oturumunda notebook'u yeniden *Tümünü çalıştır* ile çalıştırın.

In [ ]:
ok, msg = lock_is_committed([C.FINAL_PIPELINE_FILE, C.LOCK_FILE])
if not ok:
    import zipfile
    keep = ["selected_frames.csv", "data_overview.csv", "dev_val_references.csv", "experiments.csv",
            "validation_results.csv", "final_pipeline.json", "test_predictions_locked.csv"]
    with zipfile.ZipFile("kilit_commit.zip", "w") as z:
        for f in keep:
            z.write(f)
        for folder in ["seeds", "results", "figures"]:
            for root, _, files in os.walk(folder):
                for f in files:
                    z.write(os.path.join(root, f))
    try:
        from google.colab import files
        files.download("kilit_commit.zip")
    except ImportError:
        pass
    raise RuntimeError(f"⛔ Kilit kontrolü: {msg}. kilit_commit.zip içeriğini commit edin, sonra notebook'u yeniden çalıştırın.")
print("✅", msg, "– test değerlendirmesine geçilebilir.")

## 9. Bağımsız test performansı (Ödev Bölüm 13)
Test referansları **ilk kez burada** okunur. Seed 14048'in test kareleri bireysel çalışmada görülmüştü; özet **tüm kareler** ve **görülmemiş kareler** için ayrı verilir.

In [ ]:
res = pd.read_csv(C.LOCK_FILE, dtype={"sequence": str})
res.insert(5, "gt_count", [read_gt_count("02", i, lock_file=C.LOCK_FILE) for i in res.frame_idx])
seen = set()
for s in C.SEEN_TEST_SEEDS:
    seen |= set(sels[s].loc[sels[s].group == "Test", "frame_idx"])
res["gorulmus_kare"] = res.frame_idx.isin(seen)
names = [c[:-5] for c in res.columns if c.endswith("_pred")]
for n in names:
    res[f"{n}_abs_err"] = (res[f"{n}_pred"] - res["gt_count"]).abs()
res.to_csv("test_results.csv", index=False)                               # Tablo 4

summ = []
for scope, sub in [("tum_kareler", res), ("gorulmemis_kareler", res[~res.gorulmus_kare])]:
    for s in SEEDS + ["hepsi"]:
        d = sub if s == "hepsi" else sub[sub.seed == s]
        if d.empty:
            continue
        for n in names:
            m = count_metrics(d[f"{n}_pred"], d["gt_count"])
            summ.append({"kapsam": scope, "seed": s, "yontem": n, "n_kare": len(d),
                         **{k: round(v, 3) for k, v in m.items()}, "Sure_ms": round(d[f"{n}_time_ms"].mean(), 1)})
df_summ = pd.DataFrame(summ)
df_summ.to_csv("test_summary.csv", index=False)                           # Tablo 5
print("TABLO 5 – Test özeti (4 seed birleşik)"); print(df_summ[df_summ.seed == "hepsi"].to_string(index=False))
print("\nSeed bazında test MAE:")
print(df_summ[df_summ.kapsam == "tum_kareler"].pivot(index="yontem", columns="seed", values="MAE").to_string())

vt = mat[["ortalama"]].rename(columns={"ortalama": "Val_MAE_ort"})
vt.loc["M5"] = df_m5.loc[0, "Ort_Val_MAE"]
vt.index = [i if i in ("baseline", "M5") else f"M_{i}" for i in vt.index]
vt["Test_MAE"] = df_summ[(df_summ.kapsam == "tum_kareler") & (df_summ.seed == "hepsi")].set_index("yontem")["MAE"]
vt.to_csv(os.path.join(C.RESULTS_DIR, "validation_vs_test.csv"))
print("\nDoğrulama ve test karşılaştırması:"); print(vt.round(3).to_string())

## 10. Zamansal analiz (Ödev Bölüm 14)

In [ ]:
for s in SEEDS:
    d = res[res.seed == s]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
    for ax, (b, g) in zip(axes, d.groupby("block")):
        ax.plot(g.frame_idx, g.gt_count, "k-o", label="Gerçek")
        ax.plot(g.frame_idx, g.baseline_pred, "--s", color="#E53935", label="Baseline")
        ax.plot(g.frame_idx, g.M5_pred, "-^", color="#1E88E5", label="5. yöntem")
        ax.set_title(f"Seed {s} – blok {b}", fontsize=9); ax.set_xlabel("Kare"); ax.grid(alpha=0.3)
    axes[0].set_ylabel("Hücre sayısı"); axes[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/fig5_temporal_{s}.png", dpi=200); plt.show()

temporal = res[["seed", "block", "frame_idx", "gt_count", "baseline_pred", "M5_pred"]].copy()
for c in ["gt_count", "baseline_pred", "M5_pred"]:
    temporal[f"d_{c}"] = temporal.groupby(["seed", "block"])[c].diff()
temporal.to_csv(os.path.join(C.RESULTS_DIR, "temporal_analysis.csv"), index=False)
print(temporal.groupby("seed")[["d_gt_count", "d_baseline_pred", "d_M5_pred"]].agg(lambda x: x.abs().mean()).round(2).to_string())

## 11. Sonuçları indir
2. aşamanın çıktıları (test sonuçları, grafikler) için: zip'i indirip `test` commit'i olarak GitHub'a yükleyin.

In [ ]:
import zipfile
with zipfile.ZipFile("test_commit.zip", "w") as z:
    for f in ["test_results.csv", "test_summary.csv"]:
        z.write(f)
    for folder in ["results", "figures"]:
        for root, _, files_ in os.walk(folder):
            for f in files_:
                z.write(os.path.join(root, f))
try:
    from google.colab import files
    files.download("test_commit.zip")
except ImportError:
    print("test_commit.zip oluşturuldu.")